<a href="https://colab.research.google.com/github/dakshini01/Statistical-Learning-e20181/blob/main/data_analyse/data_analyse_d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

from scipy.stats import chi2_contingency, pointbiserialr, f_oneway
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OneHotEncoder, OrdinalEncoder


class DataAnalysisTool:
    def __init__(self):
        self.df = None

    # -------------------------
    # 1. Load Data
    # -------------------------
    def load_csv(self, file_path):
        self.df = pd.read_csv(
            file_path,
            na_values=["?", "N/A", "n/a", "NULL", "null", " "]
        )

        for col in self.df.columns:
            numeric_col = pd.to_numeric(self.df[col], errors="coerce")
            if not numeric_col.isna().all():
                self.df[col] = numeric_col

        print("Data loaded successfully.")
        return self.df

    # -------------------------
    # 2. Basic Summary
    # -------------------------
    def summary(self):
        if self.df is None:
            print("No data loaded.")
            return

        print("Rows:", self.df.shape[0])
        print("Columns:", self.df.shape[1])
        print("\nNumerical columns:")
        print(self.df.select_dtypes(include=np.number).columns.tolist())

        print("\nCategorical columns:")
        print(self.df.select_dtypes(exclude=np.number).columns.tolist())

        return self.df.head()

    # -------------------------
    # 3. Missing Data
    # -------------------------
    def missing_summary(self):
        if self.df is None:
            print("No data loaded.")
            return

        missing = self.df.isnull().sum()
        missing = missing[missing > 0]
        return missing

    def fill_missing(self, strategy="median", fill_value=None):
        if self.df is None:
            print("No data loaded.")
            return

        for col in self.df.columns:
            if self.df[col].isnull().sum() == 0:
                continue

            if strategy == "mean" and pd.api.types.is_numeric_dtype(self.df[col]):
                self.df[col] = self.df[col].fillna(self.df[col].mean())

            elif strategy == "median" and pd.api.types.is_numeric_dtype(self.df[col]):
                self.df[col] = self.df[col].fillna(self.df[col].median())

            elif strategy == "mode":
                self.df[col] = self.df[col].fillna(self.df[col].mode()[0])

            elif strategy == "constant":
                self.df[col] = self.df[col].fillna(fill_value)

        print("Missing values handled.")
        return self.df

    # -------------------------
    # 4. Remove Duplicates
    # -------------------------
    def remove_duplicates(self):
        before = len(self.df)
        self.df = self.df.drop_duplicates().reset_index(drop=True)
        after = len(self.df)

        print(f"Removed {before - after} duplicate rows.")
        return self.df

    # -------------------------
    # 5. Outlier Detection
    # -------------------------
    def detect_outliers(self, columns=None):
        if columns is None:
            columns = self.df.select_dtypes(include=np.number).columns

        outlier_indices = set()

        for col in columns:
            q1 = self.df[col].quantile(0.25)
            q3 = self.df[col].quantile(0.75)
            iqr = q3 - q1

            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr

            outliers = self.df[(self.df[col] < lower) | (self.df[col] > upper)]
            outlier_indices.update(outliers.index)

            print(f"{col}: {len(outliers)} outliers found")

        return self.df.loc[list(outlier_indices)]

    def remove_outliers(self, columns=None):
        outliers = self.detect_outliers(columns)
        self.df = self.df.drop(index=outliers.index).reset_index(drop=True)

        print("Outliers removed.")
        return self.df

    # -------------------------
    # 6. Encoding Categorical Data
    # -------------------------
    def encode_categorical(self, method="onehot"):
        cat_df = self.df.select_dtypes(exclude=np.number)

        if cat_df.empty:
            print("No categorical columns found.")
            return pd.DataFrame()

        if method == "onehot":
            encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
            encoded = encoder.fit_transform(cat_df.fillna("Missing"))
            cols = encoder.get_feature_names_out(cat_df.columns)
            return pd.DataFrame(encoded, columns=cols)

        elif method == "ordinal":
            encoder = OrdinalEncoder()
            encoded = encoder.fit_transform(cat_df.fillna("Missing"))
            return pd.DataFrame(encoded, columns=cat_df.columns)

    # -------------------------
    # 7. Scale Numerical Data
    # -------------------------
    def scale_numeric(self, method="standard"):
        num_df = self.df.select_dtypes(include=np.number)

        if method == "minmax":
            scaler = MinMaxScaler()
        elif method == "robust":
            scaler = RobustScaler()
        else:
            scaler = StandardScaler()

        scaled = scaler.fit_transform(num_df.fillna(num_df.median()))
        return pd.DataFrame(scaled, columns=num_df.columns)

    # -------------------------
    # 8. Create ML Dataset
    # -------------------------
    def create_ml_dataset(self, scaling="standard", encoding="onehot"):
        num_scaled = self.scale_numeric(method=scaling)
        cat_encoded = self.encode_categorical(method=encoding)

        final_df = pd.concat([num_scaled, cat_encoded], axis=1)
        return final_df

    # -------------------------
    # 9. Numerical Correlation
    # -------------------------
    def numerical_correlation(self):
        num_df = self.df.select_dtypes(include=np.number)
        corr = num_df.corr()

        fig = px.imshow(
            corr,
            text_auto=True,
            title="Numerical Correlation Heatmap"
        )
        fig.show()

        return corr

    # -------------------------
    # 10. Categorical Correlation
    # -------------------------
    def categorical_correlation(self):
        cat_df = self.df.select_dtypes(exclude=np.number)
        cols = cat_df.columns

        matrix = pd.DataFrame(
            np.zeros((len(cols), len(cols))),
            index=cols,
            columns=cols
        )

        for col1 in cols:
            for col2 in cols:
                table = pd.crosstab(cat_df[col1], cat_df[col2])

                if table.shape[0] > 1 and table.shape[1] > 1:
                    chi2 = chi2_contingency(table)[0]
                    n = table.sum().sum()
                    v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))
                else:
                    v = 0

                matrix.loc[col1, col2] = v

        fig = px.imshow(
            matrix,
            text_auto=True,
            title="Categorical Association Heatmap"
        )
        fig.show()

        return matrix

    # -------------------------
    # 11. Plots
    # -------------------------
    def plot_histogram(self, column):
        fig = px.histogram(self.df, x=column, title=f"Histogram of {column}")
        fig.show()

    def plot_boxplot(self, column):
        fig = px.box(self.df, y=column, title=f"Boxplot of {column}")
        fig.show()

    def plot_scatter(self, x, y):
        fig = px.scatter(
            self.df,
            x=x,
            y=y,
            trendline="ols",
            title=f"{x} vs {y}"
        )
        fig.show()

    def plot_bar(self, column):
        counts = self.df[column].value_counts().reset_index()
        counts.columns = [column, "count"]

        fig = px.bar(
            counts,
            x=column,
            y="count",
            title=f"Bar Chart of {column}"
        )
        fig.show()

    # -------------------------
    # 12. Export
    # -------------------------
    def export_csv(self, filename="cleaned_data.csv"):
        self.df.to_csv(filename, index=False)
        print(f"Saved as {filename}")